In [ ]:
import orekit
orekit.initVM()

In [ ]:
from orekit.pyhelpers import download_orekit_data_curdir, setup_orekit_curdir
download_orekit_data_curdir()
setup_orekit_curdir()

In [ ]:
import math
from collections import defaultdict

from java.util import ArrayList
from java.net import URL  # Add this line
from java.io import BufferedReader, InputStreamReader
from java.lang import StringBuilder

from org.hipparchus.geometry.euclidean.threed import Vector3D

from org.orekit.orbits import (
    KeplerianOrbit,
    PositionAngleType,
    WalkerConstellation,
    WalkerConstellationSlot,
    OrbitType
)
from org.orekit.frames import FramesFactory
from org.orekit.time import AbsoluteDate, TimeScalesFactory
from org.orekit.utils import Constants, IERSConventions
from org.orekit.bodies import OneAxisEllipsoid
from org.orekit.geometry.fov import CircularFieldOfView
from org.orekit.propagation.analytical import KeplerianPropagator
from org.orekit.propagation.analytical.tle import TLE, TLEPropagator  # Add this if using TLE
from org.orekit.propagation.events import (
    InterSatDirectViewDetector,
    FieldOfViewDetector,
    RelativeDistanceDetector,
    BooleanDetector,
    EventsLogger,
    VisibilityTrigger
)
from org.orekit.propagation.events.handlers import ContinueOnEvent

In [ ]:
import urllib.request
import ssl
from org.orekit.propagation.analytical.tle import TLE
from org.orekit.utils import Constants

mu = Constants.WGS84_EARTH_MU

# Function to fetch Globalstar TLEs and store with IDs
def fetch_globalstar_tles_with_ids():
    context = ssl._create_unverified_context()
    url = "https://celestrak.org/NORAD/elements/gp.php?GROUP=globalstar&FORMAT=tle"
    
    tle_data = []  # List of tuples (tle_object, globalstar_id)

    try:
        print("Fetching TLEs from CelesTrak...")
        request = urllib.request.Request(url)
        request.add_header("User-Agent", "Mozilla/5.0")

        with urllib.request.urlopen(request, context=context, timeout=30) as response:
            data = response.read().decode("utf-8")
            lines = data.splitlines()

            i = 0
            while i < len(lines):
                name = lines[i].strip()

                # TLE format: name line + line1 + line2
                if i + 2 < len(lines):
                    line1 = lines[i + 1].strip()
                    line2 = lines[i + 2].strip()

                    if line1.startswith("1 ") and line2.startswith("2 "):
                        try:
                            tle = TLE(line1, line2)
                            tle_data.append((tle, name))
                        except Exception as e:
                            print(f"✗ TLE creation failed for {name}: {e}")

                    i += 3
                else:
                    break

    except Exception as e:
        print(f"Error fetching Globalstar TLEs: {e}")

    return tle_data


# Fetch TLEs with Globalstar IDs
print("\nFetching real Globalstar TLEs...")
tle_data = fetch_globalstar_tles_with_ids()

# Separate into lists for convenience
tle_list = [item[0] for item in tle_data]
globalstar_ids = [item[1] for item in tle_data]

print(f"\n=== RESULTS ===")
print(f"Found {len(tle_list)} Globalstar satellites")

# Print all TLEs with Globalstar IDs
print("\n" + "=" * 80)
print("GLOBALSTAR CONSTELLATION - RAW TLE DATA WITH IDS")
print("=" * 80)

for i, (tle, globalstar_id) in enumerate(tle_data):
    print(f"\n{globalstar_id}")
    print(tle.getLine1())
    print(tle.getLine2())

print("\n" + "=" * 80)
print(f"Total: {len(tle_data)} satellites displayed")
print("=" * 80)

# Also print a simple list of all Globalstar IDs found
print("\nGlobalstar Satellite IDs:")
globalstar_numbers = [id_str.replace("GLOBALSTAR ", "") for id_str in globalstar_ids]
print(", ".join(globalstar_numbers))

In [ ]:
# ============================================================
# 3) BUILD TLE PROPAGATORS FOR EACH IRIDIUM SATELLITE
# ============================================================
from org.orekit.propagation.analytical.tle import TLEPropagator

global_props = {}
global_id_map = {}  # Map propagator keys to Iridium IDs

for i, tle in enumerate(tle_list):
    propagator = TLEPropagator.selectExtrapolator(tle)
    global_props[i] = propagator  # Use simple integer index
    global_id_map[i] = globalstar_ids[i]  # Store the Iridium ID

print(f"\nBuilt {len(global_props)} Globalstar TLE propagators.")

# Optional: Print first few satellites to verify
print("\nFirst 5 satellites:")
for i in range(min(5, len(global_props))):
    print(f"  Sat {i}: {global_id_map[i]}")

In [ ]:
from org.orekit.time import AbsoluteDate, TimeScalesFactory
from org.orekit.frames import FramesFactory
from org.orekit.utils import Constants, IERSConventions
from org.orekit.bodies import OneAxisEllipsoid, CelestialBodyFactory

UTC   = TimeScalesFactory.getUTC()
frame = FramesFactory.getEME2000()
mu    = Constants.WGS84_EARTH_MU

ITRF  = FramesFactory.getITRF(IERSConventions.IERS_2010, True)
earth = OneAxisEllipsoid(
    float(Constants.WGS84_EARTH_EQUATORIAL_RADIUS),
    float(Constants.WGS84_EARTH_FLATTENING),
    ITRF
)

sun  = CelestialBodyFactory.getSun()
moon = CelestialBodyFactory.getMoon()

In [ ]:
# ============================================================
# 4) DUMMY SPACECRAFT ORBIT (REPLACE LATER WITH YOUR OWN)
# ============================================================
epoch= AbsoluteDate(2026, 3, 14, 18, 59, 0.0, UTC);
sc_sma  = float(Constants.WGS84_EARTH_EQUATORIAL_RADIUS + 500000.0)  # 500 km
sc_ecc  = 0.001
sc_inc  = math.radians(37.0)
sc_raan = math.radians(20.0)
sc_aop  = 0.0
sc_ta   = math.radians(0.0)

sc_orbit = KeplerianOrbit(
    float(sc_sma),
    float(sc_ecc),
    float(sc_inc),
    float(sc_aop),
    float(sc_raan),
    float(sc_ta),
    PositionAngleType.TRUE,
    frame,
    epoch,
    float(mu)
)# ============================================================
# 4) SPACECRAFT ORBIT WITH NUMERICAL PROPAGATOR
#    Forces: J2 + Drag + Sun/Moon 3rd body
# ============================================================

from org.hipparchus.ode.nonstiff import DormandPrince853Integrator
from org.orekit.propagation import SpacecraftState
from org.orekit.propagation.numerical import NumericalPropagator
from org.orekit.orbits import OrbitType
from orekit import JArray_double
from org.orekit.forces.gravity import HolmesFeatherstoneAttractionModel, ThirdBodyAttraction
from org.orekit.forces.gravity.potential import GravityFieldFactory
from org.orekit.bodies import CelestialBodyFactory
from org.orekit.models.earth.atmosphere import HarrisPriester
from org.orekit.forces.drag import DragForce, IsotropicDrag
from org.orekit.frames import FramesFactory

# ------------------------------------------------------------
# Dummy spacecraft initial orbit (replace later with yours)
# ------------------------------------------------------------
sc_sma  = float(Constants.WGS84_EARTH_EQUATORIAL_RADIUS + 500000.0)  # 500 km
sc_ecc  = 0.001
sc_inc  = math.radians(37.0)
sc_raan = math.radians(20.0)
sc_aop  = 0.0
sc_ta   = math.radians(0.0)

sc_orbit = KeplerianOrbit(
    float(sc_sma),
    float(sc_ecc),
    float(sc_inc),
    float(sc_aop),
    float(sc_raan),
    float(sc_ta),
    PositionAngleType.TRUE,
    frame,
    epoch,
    float(mu)
)


# ------------------------------------------------------------
# Spacecraft physical properties for drag
# ------------------------------------------------------------
sc_mass = 50.0      # kg
Cd      = 2.2
area    = 15      # m^2

initial_state = SpacecraftState(sc_orbit, float(sc_mass))
sc_initial_state = initial_state
print("\nDummy spacecraft orbit created:")
print(f"  Altitude : {(sc_sma - Constants.WGS84_EARTH_EQUATORIAL_RADIUS)/1000:.1f} km")
print(f"  Period   : {sc_orbit.getKeplerianPeriod()/60:.2f} min")
print(f"  Inc      : {math.degrees(sc_inc):.1f} deg")

# ------------------------------------------------------------
# Earth model for gravity/drag occultation frame use
# ------------------------------------------------------------
ITRF = FramesFactory.getITRF(IERSConventions.IERS_2010, True)

earth = OneAxisEllipsoid(
    float(Constants.WGS84_EARTH_EQUATORIAL_RADIUS),
    float(Constants.WGS84_EARTH_FLATTENING),
    ITRF
)

# ------------------------------------------------------------
# Numerical integrator
# ------------------------------------------------------------
# dP is a position error scale used to build tolerances
dP = 10.0  # meters
orbit_type = OrbitType.CARTESIAN
tol = NumericalPropagator.tolerances(float(dP), sc_orbit, orbit_type)

min_step = 0.001
max_step = 300.0
integrator = DormandPrince853Integrator(
    float(min_step),
    float(max_step),
    JArray_double.cast_(tol[0]),
    JArray_double.cast_(tol[1])
)

# ------------------------------------------------------------
# Numerical propagator
# ------------------------------------------------------------
sc_prop = NumericalPropagator(integrator)
sc_prop.setOrbitType(orbit_type)
sc_prop.setInitialState(initial_state)
sc_prop.setMu(float(mu))

# ------------------------------------------------------------
# Force model 1: Earth gravity with J2 only
# ------------------------------------------------------------
# degree=2, order=0 => zonal J2-only gravity field
gravity_provider = GravityFieldFactory.getNormalizedProvider(2, 0)
j2_model = HolmesFeatherstoneAttractionModel(
    earth.getBodyFrame(),
    gravity_provider
)
sc_prop.addForceModel(j2_model)

# ------------------------------------------------------------
# Force model 2: Third-body perturbations (Sun and Moon)
# ------------------------------------------------------------
sun  = CelestialBodyFactory.getSun()
moon = CelestialBodyFactory.getMoon()

sun_3body  = ThirdBodyAttraction(sun)
moon_3body = ThirdBodyAttraction(moon)

sc_prop.addForceModel(sun_3body)
sc_prop.addForceModel(moon_3body)

# ------------------------------------------------------------
# Force model 3: Atmospheric drag
# ------------------------------------------------------------
# Harris-Priester is a good first choice because it does not need
# external space-weather inputs.
atmosphere = HarrisPriester(sun, earth)

drag_spacecraft = IsotropicDrag(float(area), float(Cd))
drag_model = DragForce(atmosphere, drag_spacecraft)

sc_prop.addForceModel(drag_model)

print("\nNumerical spacecraft propagator configured with:")
print("  - Earth gravity: J2 only")
print("  - Third body: Sun")
print("  - Third body: Moon")
print("  - Drag: Harris-Priester + IsotropicDrag")
print(f"  - Mass = {sc_mass:.2f} kg, Cd = {Cd:.2f}, Area = {area:.3f} m^2")

# ============================================================
# 5) EARTH MODEL FOR OCCULTATION
# ============================================================
ITRF = FramesFactory.getITRF(IERSConventions.IERS_2010, True)

earth = OneAxisEllipsoid(
    float(Constants.WGS84_EARTH_EQUATORIAL_RADIUS),
    float(Constants.WGS84_EARTH_FLATTENING),
    ITRF
)

# ============================================================
# 6) ANTENNA FOV MODEL
# ============================================================
half_aperture_deg = 60.0
half_aperture_rad = math.radians(half_aperture_deg)

boresight = Vector3D.PLUS_K
fov_margin = 0.0

antenna_fov = CircularFieldOfView(
    boresight,
    float(half_aperture_rad),
    float(fov_margin)
)

# ============================================================
# 7) MAX RANGE
# ============================================================
max_range_m = 2_500_000.0   # 2500 km

# ============================================================
# 8) SET GLOBALSTAR SATELLITE ATTITUDE
#    Approximation: nadir-pointing antenna boresight
# ============================================================
from org.orekit.attitudes import NadirPointing

for sat_key, sat_prop in global_props.items():
    sat_prop.setAttitudeProvider(NadirPointing(frame, earth))

print(f"Set nadir-pointing attitude for {len(global_props)} Globalstar propagators.")

# ============================================================
# 9) BUILD SPACECRAFT EPHEMERIS ONCE
# ============================================================
start_date = epoch
end_date   = epoch.shiftedBy(24.0 * 3600.0)

ephem_gen = sc_prop.getEphemerisGenerator()

print("\nPropagating spacecraft once to build ephemeris...")
sc_final_state = sc_prop.propagate(start_date, end_date)
sc_ephem = ephem_gen.getGeneratedEphemeris()

sc_ephem_start = start_date
sc_ephem_end   = sc_final_state.getDate()

print("Spacecraft ephemeris generated")
print(f"  Start : {sc_ephem_start}")
print(f"  End   : {sc_ephem_end}")

# ============================================================
# 10) OPTIONAL DEBUG AT INITIAL EPOCH FOR ONE SATELLITE
# ============================================================
test_key = 0
sat_prop_test = global_props[test_key]

los_test = InterSatDirectViewDetector(earth, sc_ephem)

raw_fov_test = FieldOfViewDetector(
    sc_ephem,
    0.0,
    VisibilityTrigger.VISIBLE_ONLY_WHEN_FULLY_IN_FOV,
    antenna_fov
)
fov_inside_test = BooleanDetector.notCombine(raw_fov_test)

raw_range_test = RelativeDistanceDetector(sc_ephem, float(max_range_m))
range_within_test = BooleanDetector.notCombine(raw_range_test)

sat_initial_state_test = sat_prop_test.propagate(sc_ephem_start)

print(f"\nInitial detector debug for satellite {test_key} ({global_id_map[test_key]}):")
print("  LOS g           =", los_test.g(sat_initial_state_test))
print("  inside-FOV g    =", fov_inside_test.g(sat_initial_state_test))
print("  within-range g  =", range_within_test.g(sat_initial_state_test))

# ============================================================
# 11) CONTACT ANALYSIS: GLOBALSTAR SATELLITE AS OBSERVER
# ============================================================
contact_windows = []

print("\n================ CONTACT ANALYSIS ================\n")
print(f"FOV half-angle : {half_aperture_deg:.1f} deg")
print(f"Max range      : {max_range_m/1000.0:.1f} km")
print(f"Start          : {sc_ephem_start}")
print(f"End            : {sc_ephem_end}\n")

for sat_key, sat_prop in global_props.items():

    logger = EventsLogger()

    # --------------------------------------------------------
    # 1) Get satellite state at analysis start FIRST
    #    Do this before attaching any detector that uses sc_ephem
    # --------------------------------------------------------
    sat_initial_state = sat_prop.propagate(sc_ephem_start)

    # --------------------------------------------------------
    # 2) Build detectors using spacecraft ephemeris as target
    # --------------------------------------------------------
    los_detector = InterSatDirectViewDetector(earth, sc_ephem)

    raw_fov_detector = FieldOfViewDetector(
        sc_ephem,
        0.0,
        VisibilityTrigger.VISIBLE_ONLY_WHEN_FULLY_IN_FOV,
        antenna_fov
    )
    fov_inside_detector = BooleanDetector.notCombine(raw_fov_detector)

    raw_range_detector = RelativeDistanceDetector(sc_ephem, float(max_range_m))
    range_within_detector = BooleanDetector.notCombine(raw_range_detector)

    combined_detector = BooleanDetector.andCombine([
        los_detector,
        fov_inside_detector,
        range_within_detector
    ]).withHandler(ContinueOnEvent()) \
     .withMaxCheck(60.0) \
     .withThreshold(1.0e-6)
    
    
    g_los = los_detector.g(sat_initial_state)
    g_fov = fov_inside_detector.g(sat_initial_state)
    g_rng = range_within_detector.g(sat_initial_state)

    
    in_contact = (g_los > 0.0) and (g_fov > 0.0) and (g_rng > 0.0)
    current_start = sc_ephem_start if in_contact else None

    
    monitored_detector = logger.monitorDetector(combined_detector)
    sat_prop.addEventDetector(monitored_detector)

    # --------------------------------------------------------
    # 5) Propagate satellite only over valid spacecraft-ephem interval
    # --------------------------------------------------------
    _ = sat_prop.propagate(sc_ephem_start, sc_ephem_end)

    # --------------------------------------------------------
    # 6) Collect events for this satellite
    # --------------------------------------------------------
    sat_events = []
    for ev in logger.getLoggedEvents():
        sat_events.append({
            "date": ev.getState().getDate(),
            "increasing": ev.isIncreasing()
        })

    sat_events.sort(key=lambda e: e["date"].durationFrom(sc_ephem_start))

    # --------------------------------------------------------
    # 7) Convert events to contact windows
    # --------------------------------------------------------
    for ev in sat_events:
        ev_date = ev["date"]
        increasing = ev["increasing"]

        if increasing:
            if not in_contact:
                current_start = ev_date
                in_contact = True
        else:
            if in_contact and current_start is not None:
                duration_s = ev_date.durationFrom(current_start)
                if duration_s >= 0.0:
                    contact_windows.append({
                        "sat_index": sat_key,
                        "global_id": global_id_map[sat_key],
                        "start": current_start,
                        "end": ev_date,
                        "duration_s": duration_s
                    })
                current_start = None
                in_contact = False

    # --------------------------------------------------------
    # 8) Open window until analysis end
    # --------------------------------------------------------
    if in_contact and current_start is not None:
        duration_s = sc_ephem_end.durationFrom(current_start)
        if duration_s >= 0.0:
            contact_windows.append({
                "sat_index": sat_key,
                "global_id": global_id_map[sat_key],
                "start": current_start,
                "end": sc_ephem_end,
                "duration_s": duration_s
            })
# ============================================================
# 12) PRINT CONTACT WINDOWS
# ============================================================
print("\n================ CONTACT WINDOWS ================\n")
print(f"FOV half-angle : {half_aperture_deg:.1f} deg")
print(f"Max range      : {max_range_m/1000.0:.1f} km\n")

if len(contact_windows) == 0:
    print("No contact windows found in the analysis interval.")
else:
    print(f"{'Idx':>4} {'Globalstar ID':>15} {'Start UTC':>28} {'End UTC':>28} {'Dur (min)':>10}")
    print("-" * 95)

    for w in contact_windows:
        print(f"{w['sat_index']:>4} "
              f"{w['global_id']:>15} "
              f"{str(w['start']):>28} "
              f"{str(w['end']):>28} "
              f"{w['duration_s']/60.0:>10.2f}")

# ============================================================
# 13) SUMMARY PER SATELLITE
# ============================================================
summary = defaultdict(lambda: {"count": 0, "total_s": 0.0})

for w in contact_windows:
    key = w["sat_index"]
    summary[key]["count"] += 1
    summary[key]["total_s"] += w["duration_s"]

print("\n================ CONTACT SUMMARY ================\n")
print(f"{'Idx':>4} {'Globalstar ID':>15} {'#Windows':>9} {'Total (min)':>12} {'Total (hr)':>11}")
print("-" * 60)

for key in sorted(summary.keys()):
    count = summary[key]["count"]
    total_min = summary[key]["total_s"] / 60.0
    total_hr  = summary[key]["total_s"] / 3600.0
    print(f"{key:>4} "
          f"{global_id_map[key]:>15} "
          f"{count:>9} "
          f"{total_min:>12.2f} "
          f"{total_hr:>11.2f}")

print(f"\nTotal contact windows found: {len(contact_windows)}")

# ============================================================
# 14) DEBUG RANGE AT CONTACT START (SAME FRAME)
# ============================================================
if len(contact_windows) > 0:
    print("\n========== DEBUG RANGE AT CONTACT START ==========\n")

    for w in contact_windows:
        key = w["sat_index"]
        t0  = w["start"]

        sat_prop = global_props[key]

        sat_state = sat_prop.propagate(t0)
        sc_state  = sc_ephem.propagate(t0)

        r_sc  = sc_state.getPVCoordinates(frame).getPosition()
        r_sat = sat_state.getPVCoordinates(frame).getPosition()

        dx = r_sat.getX() - r_sc.getX()
        dy = r_sat.getY() - r_sc.getY()
        dz = r_sat.getZ() - r_sc.getZ()

        rng_km = math.sqrt(dx*dx + dy*dy + dz*dz) / 1000.0

        print(f"Sat {key:>3} ({global_id_map[key]}): "
              f"contact-start range = {rng_km:.2f} km at {t0}")